<a href="https://colab.research.google.com/github/zeynepdnnz/cs445-semeval-task5/blob/main/CS445_Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
homonym                                                     potential
judged_meaning      the difference in electrical charge between tw...
precontext          The old machine hummed in the corner of the wo...
sentence                          The potential couldn't be measured.
ending              She collected a battery reader and looked on e...
choices                                               [4, 5, 2, 3, 1]
average                                                           3.0
stdev                                                        1.581139
nonsensical                       [False, False, False, False, False]
sample_id                                                        1843
example_sentence         The circuit has a high potential difference.
'''

"\nhomonym                                                     potential\njudged_meaning      the difference in electrical charge between tw...\nprecontext          The old machine hummed in the corner of the wo...\nsentence                          The potential couldn't be measured.\nending              She collected a battery reader and looked on e...\nchoices                                               [4, 5, 2, 3, 1]\naverage                                                           3.0\nstdev                                                        1.581139\nnonsensical                       [False, False, False, False, False]\nsample_id                                                        1843\nexample_sentence         The circuit has a high potential difference.\n"

In [ ]:
!pip install -q "transformers>=4.40,<4.45"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Primary Dataset Fine-Tune Baseline**

In [ ]:
import pandas
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, BatchEncoding,
    DataCollatorWithPadding, EvalPrediction,
    EarlyStoppingCallback
    )
from datasets import Dataset, DatasetDict
import numpy as np
from scipy.stats import spearmanr

In [ ]:
ambistory_train_df = pandas.read_json("/content/train.json").T
ambistory_validation_df = pandas.read_json("/content/dev.json").T
ambistory_test_df = pandas.read_json("/content/test_labeled.json").T

In [ ]:
model_id = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def input_format(sample: pandas.Series) -> tuple[str, str]:
  story_part = [sample['precontext'], sample['sentence'], sample['ending'] or '']
  story_part = " ".join(story_part)
  meaning_part = (f"{sample['homonym']}: {sample['judged_meaning']} "
                   f"(e.g., \"{sample['example_sentence']}\")")

  return story_part, meaning_part


def tokenize_samples(batch: dict) -> BatchEncoding:
    story_parts = []
    meaning_parts = []
    for i in range(len(batch["precontext"])):
        ending = batch["ending"][i] or ''
        story = f"{batch['precontext'][i]} {batch['sentence'][i]} {ending}"
        meaning = (
            f"{batch['homonym'][i]}: {batch['judged_meaning'][i]} "
            f'(e.g., "{batch["example_sentence"][i]}")'
        )
        story_parts.append(story)
        meaning_parts.append(meaning)

    return tokenizer(
        story_parts,
        meaning_parts,
        truncation="only_first",
        max_length=256,
        padding=False,
    )


def metric_computation(eval_pred: EvalPrediction) -> dict[str, float]:
    predictions, labels = eval_pred
    predictions = predictions.squeeze()

    spearman_score, _ = spearmanr(predictions, labels)


    if np.isnan(spearman_score):
        spearman_score = 0.0

    mean_absolute_error = np.mean(np.abs(predictions - labels))
    root_mean_squared_error = np.sqrt(np.mean((predictions - labels) ** 2))

    return {
        "spearman": float(spearman_score),
        "mae": float(mean_absolute_error),
        "rmse": float(root_mean_squared_error),
    }
def compute_acc_within_std(trainer, dataset, original_df):
    preds = trainer.predict(dataset).predictions.squeeze()
    preds = np.clip(preds, 1.0, 5.0)

    labels = original_df["average"].to_numpy(dtype=float)
    stdev = original_df["stdev"].to_numpy(dtype=float)

    threshold = np.maximum(stdev, 1.0)
    acc_within_std = float(np.mean(np.abs(preds - labels) <= threshold))

    return acc_within_std

In [ ]:
def evaluate_labeled_split(trainer, dataset, original_df, split_name="Split"):
    preds = trainer.predict(dataset).predictions.squeeze()
    preds = np.clip(preds, 1.0, 5.0)

    labels = original_df["average"].to_numpy(dtype=float)
    stdev = original_df["stdev"].to_numpy(dtype=float)

    spearman_score, _ = spearmanr(preds, labels)

    if np.isnan(spearman_score):
        spearman_score = 0.0

    mae = float(np.mean(np.abs(preds - labels)))
    rmse = float(np.sqrt(np.mean((preds - labels) ** 2)))

    threshold = np.maximum(stdev, 1.0)
    acc_sd = float(np.mean(np.abs(preds - labels) <= threshold))

    print(f"\n{split_name} metrics")
    print(f"{split_name} Spearman: {spearman_score:.4f}")
    print(f"{split_name} MAE:      {mae:.4f}")
    print(f"{split_name} RMSE:     {rmse:.4f}")
    print(f"{split_name} Acc-SD:   {acc_sd:.4f}")

    return {
        "spearman": float(spearman_score),
        "mae": mae,
        "rmse": rmse,
        "acc_sd": acc_sd,
        "preds": preds,
    }

In [ ]:
def compute_metrics_from_preds(preds, original_df):
    preds = np.clip(preds, 1.0, 5.0)

    labels = original_df["average"].to_numpy(dtype=float)
    stdev = original_df["stdev"].to_numpy(dtype=float)

    rho, _ = spearmanr(preds, labels)
    if np.isnan(rho):
        rho = 0.0

    mae = float(np.mean(np.abs(preds - labels)))
    rmse = float(np.sqrt(np.mean((preds - labels) ** 2)))

    threshold = np.maximum(stdev, 1.0)
    acc_sd = float(np.mean(np.abs(preds - labels) <= threshold))

    return {
        "rho": float(rho),
        "mae": mae,
        "rmse": rmse,
        "acc_sd": acc_sd,
    }

In [ ]:
from datasets import Dataset, DatasetDict, Value
import pandas as pd


ambistory_train_df["average"] = pd.to_numeric(
    ambistory_train_df["average"],
    errors="coerce"
)

ambistory_validation_df["average"] = pd.to_numeric(
    ambistory_validation_df["average"],
    errors="coerce"
)
ambistory_test_df["average"] = pd.to_numeric(
    ambistory_test_df["average"],
    errors="coerce"
)


ambistory_train_df = ambistory_train_df.dropna(subset=["average","stdev"]).reset_index(drop=True)
ambistory_validation_df = ambistory_validation_df.dropna(subset=["average","stdev"]).reset_index(drop=True)
ambistory_test_df = ambistory_test_df.dropna(
    subset=["average", "stdev"]
).reset_index(drop=True)

raw_datasets = DatasetDict({
    "ambistory_train_set": Dataset.from_pandas(ambistory_train_df),
    "ambistory_validation_set": Dataset.from_pandas(ambistory_validation_df),
    "ambistory_test_set": Dataset.from_pandas(ambistory_test_df)
})


for split_name in ["ambistory_train_set", "ambistory_validation_set","ambistory_test_set"]:
    raw_datasets[split_name] = raw_datasets[split_name].rename_column("average", "labels")
    raw_datasets[split_name] = raw_datasets[split_name].cast_column("labels", Value("float32"))

print(raw_datasets)
print(raw_datasets["ambistory_train_set"].column_names)
print(raw_datasets["ambistory_validation_set"].column_names)
print(raw_datasets["ambistory_test_set"].column_names)

Casting the dataset:   0%|          | 0/2280 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/588 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/930 [00:00<?, ? examples/s]

DatasetDict({
    ambistory_train_set: Dataset({
        features: ['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence'],
        num_rows: 2280
    })
    ambistory_validation_set: Dataset({
        features: ['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence'],
        num_rows: 588
    })
    ambistory_test_set: Dataset({
        features: ['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence'],
        num_rows: 930
    })
})
['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence']
['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence']
['homonym

In [ ]:
def tokenize_dataset(dataset):
    cols_to_remove = [
        c for c in dataset.column_names
        if c != "labels"
    ]

    return dataset.map(
        tokenize_samples,
        batched=True,
        remove_columns=cols_to_remove
    )

tokenized_datasets = DatasetDict()

for split_name in raw_datasets:
    tokenized_datasets[split_name] = tokenize_dataset(raw_datasets[split_name])

Map:   0%|          | 0/2280 [00:00<?, ? examples/s]

Map:   0%|          | 0/588 [00:00<?, ? examples/s]

Map:   0%|          | 0/930 [00:00<?, ? examples/s]

In [ ]:
import json
import gc
import torch
import numpy as np
from itertools import product
from pathlib import Path


CONFIGS = [
    {"name": "E_short_warmup", "lr": 1e-5, "bs": 8, "epochs": 5, "warmup": 0.06},
]

SEEDS = [42, 1337, 2024]

from torch.optim import AdamW

def build_llrd_optimizer(model, base_lr=1e-5, layer_decay=0.85, weight_decay=0.01):
    """
    Layer-wise learning rate decay for DeBERTa.
    Upper layers get higher LR, lower layers get lower LR.
    Classifier/head gets base_lr.
    """

    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]


    layers = model.deberta.encoder.layer
    num_layers = len(layers)

    optimizer_grouped_parameters = []


    embedding_lr = base_lr * (layer_decay ** num_layers)

    embedding_params_decay = []
    embedding_params_no_decay = []

    for name, param in model.deberta.embeddings.named_parameters():
        full_name = f"deberta.embeddings.{name}"
        if not param.requires_grad:
            continue

        if any(nd in full_name for nd in no_decay):
            embedding_params_no_decay.append(param)
        else:
            embedding_params_decay.append(param)

    optimizer_grouped_parameters.append({
        "params": embedding_params_decay,
        "lr": embedding_lr,
        "weight_decay": weight_decay,
    })

    optimizer_grouped_parameters.append({
        "params": embedding_params_no_decay,
        "lr": embedding_lr,
        "weight_decay": 0.0,
    })




    for layer_idx, layer in enumerate(layers):
        layer_lr = base_lr * (layer_decay ** (num_layers - 1 - layer_idx))

        decay_params = []
        no_decay_params = []

        for name, param in layer.named_parameters():
            full_name = f"deberta.encoder.layer.{layer_idx}.{name}"
            if not param.requires_grad:
                continue

            if any(nd in full_name for nd in no_decay):
                no_decay_params.append(param)
            else:
                decay_params.append(param)

        optimizer_grouped_parameters.append({
            "params": decay_params,
            "lr": layer_lr,
            "weight_decay": weight_decay,
        })

        optimizer_grouped_parameters.append({
            "params": no_decay_params,
            "lr": layer_lr,
            "weight_decay": 0.0,
        })


    head_params_decay = []
    head_params_no_decay = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue


        if name.startswith("deberta.embeddings") or name.startswith("deberta.encoder"):
            continue

        if any(nd in name for nd in no_decay):
            head_params_no_decay.append(param)
        else:
            head_params_decay.append(param)

    optimizer_grouped_parameters.append({
        "params": head_params_decay,
        "lr": base_lr,
        "weight_decay": weight_decay,
    })

    optimizer_grouped_parameters.append({
        "params": head_params_no_decay,
        "lr": base_lr,
        "weight_decay": 0.0,
    })

    optimizer = AdamW(
        optimizer_grouped_parameters,
        lr=base_lr,
        eps=1e-8
    )

    return optimizer
def train_loop(config, seed):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=1,
        problem_type="regression",
        ignore_mismatched_sizes=True,
    )
    optimizer = build_llrd_optimizer(
      model,
      base_lr=config["lr"],
      layer_decay=0.85,
      weight_decay=0.01
    )

    args = TrainingArguments(
        output_dir=f"./sweep/{config['name']}_seed{seed}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=config["lr"],
        per_device_train_batch_size=config["bs"],
        per_device_eval_batch_size=16,
        num_train_epochs=config["epochs"],
        weight_decay=0.01,
        warmup_ratio=config["warmup"],
        bf16=False,
        fp16=False,
        load_best_model_at_end=False,
        logging_steps=200,
        report_to="none",
        seed=seed,
        disable_tqdm=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=data_collator,
        train_dataset=tokenized_datasets["ambistory_train_set"],
        eval_dataset=tokenized_datasets["ambistory_validation_set"],
        tokenizer=tokenizer,
        compute_metrics=metric_computation,
        optimizers=(optimizer, None),
    )

    trainer.train()

    eval_logs = [log for log in trainer.state.log_history if "eval_spearman" in log]
    best_eval = max(eval_logs, key=lambda x: x["eval_spearman"])

    # Evaluate the trained seed model on labeled test split
    test_preds = trainer.predict(
        tokenized_datasets["ambistory_test_set"]
    ).predictions.squeeze()

    test_preds = np.clip(test_preds, 1.0, 5.0)
    test_metrics = compute_metrics_from_preds(test_preds, ambistory_test_df)

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        # Dev metrics used for model/config selection
        "dev_spearman": best_eval["eval_spearman"],
        "dev_mae": best_eval["eval_mae"],
        "dev_rmse": best_eval["eval_rmse"],
        "best_epoch": best_eval["epoch"],

        # Test metrics for report table
        "test_rho": test_metrics["rho"],
        "test_acc_sd": test_metrics["acc_sd"],
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],

        # Needed for ensemble
        "test_preds": test_preds.tolist(),
    }


results = {}

for config in CONFIGS:
    print(f"\n{'='*60}")
    print(f"Config: {config['name']}  |  {config}")
    print('='*60)

    seed_results = []
    for seed in SEEDS:
        print(f"\n  seed={seed} ...", end=" ", flush=True)
        metrics = train_loop(config, seed)
        print(
            f"dev_spearman={metrics['dev_spearman']:.4f}  "
            f"test_rho={metrics['test_rho']:.4f}  "
            f"test_acc_sd={metrics['test_acc_sd']:.4f}  "
            f"best_epoch={metrics['best_epoch']:.0f}"
        )
        seed_results.append(metrics)

    dev_spearmans = [r["dev_spearman"] for r in seed_results]
    dev_maes = [r["dev_mae"] for r in seed_results]
    dev_rmses = [r["dev_rmse"] for r in seed_results]

    test_rhos = [r["test_rho"] for r in seed_results]
    test_acc_sds = [r["test_acc_sd"] for r in seed_results]
    test_maes = [r["test_mae"] for r in seed_results]
    test_rmses = [r["test_rmse"] for r in seed_results]
    all_test_preds = np.array([r["test_preds"] for r in seed_results])
    ensemble_test_preds = np.mean(all_test_preds, axis=0)

    ensemble_test_metrics = compute_metrics_from_preds(
        ensemble_test_preds,
        ambistory_test_df
    )
    results[config["name"]] = {
        "config": config,
        "per_seed": seed_results,

        # Dev selection metrics
        "dev_spearman_mean": float(np.mean(dev_spearmans)),
        "dev_spearman_std": float(np.std(dev_spearmans)),
        "dev_mae_mean": float(np.mean(dev_maes)),
        "dev_rmse_mean": float(np.mean(dev_rmses)),

        # Test table metrics: per-seed mean ± std
        "test_rho_mean": float(np.mean(test_rhos)),
        "test_rho_std": float(np.std(test_rhos)),
        "test_acc_sd_mean": float(np.mean(test_acc_sds)),
        "test_acc_sd_std": float(np.std(test_acc_sds)),
        "test_mae_mean": float(np.mean(test_maes)),
        "test_rmse_mean": float(np.mean(test_rmses)),

        # Test table metrics: ensemble
        "ensemble_test_rho": ensemble_test_metrics["rho"],
        "ensemble_test_acc_sd": ensemble_test_metrics["acc_sd"],
        "ensemble_test_mae": ensemble_test_metrics["mae"],
        "ensemble_test_rmse": ensemble_test_metrics["rmse"],
    }

    print(f"\n  -> Dev Spearman: {np.mean(dev_spearmans):.4f} ± {np.std(dev_spearmans):.4f}")

    print(f"  -> Test ρ:       {np.mean(test_rhos):.4f} ± {np.std(test_rhos):.4f}")
    print(f"  -> Test Acc-SD:  {np.mean(test_acc_sds):.4f} ± {np.std(test_acc_sds):.4f}")
    print(f"  -> Test MAE:     {np.mean(test_maes):.4f}")
    print(f"  -> Test RMSE:    {np.mean(test_rmses):.4f}")

    print(f"  -> Ensemble Test ρ:      {ensemble_test_metrics['rho']:.4f}")
    print(f"  -> Ensemble Test Acc-SD: {ensemble_test_metrics['acc_sd']:.4f}")
    Path("./sweep").mkdir(exist_ok=True)
    with open("./sweep/results.json", "w") as f:
        json.dump(results, f, indent=2)


print("\n\n" + "="*60)
print("SWEEP SUMMARY (sorted by mean dev Spearman)")
print("="*60)

ranked = sorted(results.items(), key=lambda x: -x[1]["dev_spearman_mean"])
print(f"\n{'Config':<20} {'Spearman':<20} {'MAE':<10} {'RMSE':<10}")
print("-" * 60)
for name, r in ranked:
    spearman_str = f"{r['dev_spearman_mean']:.4f} ± {r['dev_spearman_std']:.4f}"
    print(
        f"{name:<20} "
        f"{spearman_str:<20} "
        f"{r['dev_mae_mean']:<10.4f} "
        f"{r['dev_rmse_mean']:<10.4f}"
    )
best = ranked[0]
print(f"\n Best config: {best[0]}")
print(f"   {best[1]['config']}")
print(f"   Dev Spearman: {best[1]['dev_spearman_mean']:.4f} ± {best[1]['dev_spearman_std']:.4f}")


Config: E_short_warmup  |  {'name': 'E_short_warmup', 'lr': 1e-05, 'bs': 8, 'epochs': 5, 'warmup': 0.06}

  seed=42 ... 

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 3.0761, 'grad_norm': 60.495567321777344, 'learning_rate': 1.8510141078904075e-07, 'epoch': 0.7017543859649122}
{'eval_loss': 1.0952097177505493, 'eval_spearman': 0.5445274394809446, 'eval_mae': 0.8149104714393616, 'eval_rmse': 1.046522617340088, 'eval_runtime': 3.7565, 'eval_samples_per_second': 156.529, 'eval_steps_per_second': 9.85, 'epoch': 1.0}
{'loss': 0.8993, 'grad_norm': 21.31075096130371, 'learning_rate': 1.5488077229287082e-07, 'epoch': 1.4035087719298245}
{'eval_loss': 1.1236175298690796, 'eval_spearman': 0.5891254754326174, 'eval_mae': 0.814306378364563, 'eval_rmse': 1.0600082874298096, 'eval_runtime': 3.7701, 'eval_samples_per_second': 155.964, 'eval_steps_per_second': 9.814, 'epoch': 2.0}
{'loss': 0.7687, 'grad_norm': 16.102428436279297, 'learning_rate': 1.246601337967009e-07, 'epoch': 2.1052631578947367}
{'loss': 0.618, 'grad_norm': 43.5877799987793, 'learning_rate': 9.443949530053098e-08, 'epoch': 2.807017543859649}
{'eval_loss': 1.1215951442718506, 'eval_spearm

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 2.8889, 'grad_norm': 55.73369598388672, 'learning_rate': 1.8510141078904075e-07, 'epoch': 0.7017543859649122}
{'eval_loss': 1.1165580749511719, 'eval_spearman': 0.5461581945800131, 'eval_mae': 0.827248215675354, 'eval_rmse': 1.0566731691360474, 'eval_runtime': 3.7706, 'eval_samples_per_second': 155.941, 'eval_steps_per_second': 9.813, 'epoch': 1.0}
{'loss': 0.9141, 'grad_norm': 25.588661193847656, 'learning_rate': 1.5488077229287082e-07, 'epoch': 1.4035087719298245}
{'eval_loss': 1.181618571281433, 'eval_spearman': 0.5729573691175359, 'eval_mae': 0.8313133716583252, 'eval_rmse': 1.0870227813720703, 'eval_runtime': 3.7595, 'eval_samples_per_second': 156.405, 'eval_steps_per_second': 9.842, 'epoch': 2.0}
{'loss': 0.7796, 'grad_norm': 32.704322814941406, 'learning_rate': 1.246601337967009e-07, 'epoch': 2.1052631578947367}
{'loss': 0.6103, 'grad_norm': 39.39357376098633, 'learning_rate': 9.443949530053098e-08, 'epoch': 2.807017543859649}
{'eval_loss': 1.1039493083953857, 'eval_spe

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 3.3125, 'grad_norm': 30.43012809753418, 'learning_rate': 1.8510141078904075e-07, 'epoch': 0.7017543859649122}
{'eval_loss': 1.1975175142288208, 'eval_spearman': 0.5241389081196665, 'eval_mae': 0.8444250822067261, 'eval_rmse': 1.0943114757537842, 'eval_runtime': 3.7609, 'eval_samples_per_second': 156.347, 'eval_steps_per_second': 9.838, 'epoch': 1.0}
{'loss': 0.9243, 'grad_norm': 23.81796646118164, 'learning_rate': 1.5488077229287082e-07, 'epoch': 1.4035087719298245}
{'eval_loss': 1.2359325885772705, 'eval_spearman': 0.5884679962988671, 'eval_mae': 0.8357413411140442, 'eval_rmse': 1.111725091934204, 'eval_runtime': 3.7637, 'eval_samples_per_second': 156.23, 'eval_steps_per_second': 9.831, 'epoch': 2.0}
{'loss': 0.7696, 'grad_norm': 53.03065490722656, 'learning_rate': 1.246601337967009e-07, 'epoch': 2.1052631578947367}
{'loss': 0.6335, 'grad_norm': 28.726699829101562, 'learning_rate': 9.443949530053098e-08, 'epoch': 2.807017543859649}
{'eval_loss': 1.2555047273635864, 'eval_spea

In [ ]:
best_name = ranked[0][0]
best_config = results[best_name]["config"]

print("\n" + "="*60)
print("REPORT TABLE VALUES")
print("="*60)

r = results[best_name]

print("System: MSE + LLRD")
print(
    f"Per-seed Test ρ:      "
    f"{r['test_rho_mean']:.4f} ± {r['test_rho_std']:.4f}"
)
print(
    f"Per-seed Test Acc-SD: "
    f"{r['test_acc_sd_mean']:.4f} ± {r['test_acc_sd_std']:.4f}"
)
print(f"Ensemble Test ρ:      {r['ensemble_test_rho']:.4f}")
print(f"Ensemble Test Acc-SD: {r['ensemble_test_acc_sd']:.4f}")
print(f"\nTraining final model with config: {best_name}")
print(f"  {best_config}")

seed_results = results[best_name]["per_seed"]
mean_spearman = results[best_name]["dev_spearman_mean"]
representative_idx = np.argmin([
    abs(r["dev_spearman"] - mean_spearman)
    for r in seed_results
])
final_seed = SEEDS[representative_idx]
print(f"  using seed={final_seed} (closest to mean)")

final_model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    problem_type="regression",
    ignore_mismatched_sizes=True,
)
final_optimizer = build_llrd_optimizer(
    final_model,
    base_lr=best_config["lr"],
    layer_decay=0.85,
    weight_decay=0.01
)

final_args = TrainingArguments(
    output_dir="./baseline_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_config["lr"],
    per_device_train_batch_size=best_config["bs"],
    per_device_eval_batch_size=16,
    num_train_epochs=best_config["epochs"],
    weight_decay=0.01,
    warmup_ratio=best_config["warmup"],
    bf16=False,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    seed=final_seed,
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["ambistory_train_set"],
    eval_dataset=tokenized_datasets["ambistory_validation_set"],
    tokenizer=tokenizer,
    compute_metrics=metric_computation,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    optimizers=(final_optimizer, None),
)

final_trainer.train()
print("\n" + "="*60)
print("FINAL DEV EVALUATION")
print("="*60)

dev_metrics = evaluate_labeled_split(
    final_trainer,
    tokenized_datasets["ambistory_validation_set"],
    ambistory_validation_df,
    split_name="Dev"
)

print("\n" + "="*60)
print("FINAL TEST EVALUATION")
print("="*60)

test_metrics = evaluate_labeled_split(
    final_trainer,
    tokenized_datasets["ambistory_test_set"],
    ambistory_test_df,
    split_name="Test"
)

test_preds = test_metrics["preds"]

submission_df = pd.DataFrame({
    "sample_id": ambistory_test_df["sample_id"].values,
    "prediction": test_preds
})

Path("./baseline_final").mkdir(exist_ok=True)
submission_df.to_csv("./baseline_final/submission.csv", index=False)

print(submission_df.head())
print("\nSaved predictions to ./baseline_final/submission.csv")
final_trainer.save_model("./baseline_final/best_model")

with open("./baseline_final/run_info.json", "w") as f:
    json.dump({
        "config": best_config,
        "seed": final_seed,

        "dev_spearman_mean_over_seeds": results[best_name]["dev_spearman_mean"],
        "dev_spearman_std_over_seeds": results[best_name]["dev_spearman_std"],
        "dev_mae_mean_over_seeds": results[best_name]["dev_mae_mean"],
        "dev_rmse_mean_over_seeds": results[best_name]["dev_rmse_mean"],

        "test_rho_mean_over_seeds": results[best_name]["test_rho_mean"],
        "test_rho_std_over_seeds": results[best_name]["test_rho_std"],
        "test_acc_sd_mean_over_seeds": results[best_name]["test_acc_sd_mean"],
        "test_acc_sd_std_over_seeds": results[best_name]["test_acc_sd_std"],

        "ensemble_test_rho": results[best_name]["ensemble_test_rho"],
        "ensemble_test_acc_sd": results[best_name]["ensemble_test_acc_sd"],

        "final_dev_spearman": dev_metrics["spearman"],
        "final_dev_mae": dev_metrics["mae"],
        "final_dev_rmse": dev_metrics["rmse"],
        "final_dev_acc_sd": dev_metrics["acc_sd"],

        "final_test_spearman": test_metrics["spearman"],
        "final_test_mae": test_metrics["mae"],
        "final_test_rmse": test_metrics["rmse"],
        "final_test_acc_sd": test_metrics["acc_sd"],

        "note": "LLRD ablation evaluated on labeled dev and labeled test_labeled.json splits."
    }, f, indent=2)

print(f"\nSaved to ./baseline_final/")


REPORT TABLE VALUES
System: MSE + LLRD
Per-seed Test ρ:      0.6041 ± 0.0053
Per-seed Test Acc-SD: 0.7434 ± 0.0013
Ensemble Test ρ:      0.6141
Ensemble Test Acc-SD: 0.7387

Training final model with config: E_short_warmup
  {'name': 'E_short_warmup', 'lr': 1e-05, 'bs': 8, 'epochs': 5, 'warmup': 0.06}
  using seed=42 (closest to mean)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Spearman,Mae,Rmse
1,1.039400,1.095210,0.544527,0.814910,1.046523


Epoch,Training Loss,Validation Loss,Spearman,Mae,Rmse
1,1.039400,1.095210,0.544527,0.814910,1.046523
2,0.800300,1.123618,0.589125,0.814306,1.060008
3,0.602700,1.121595,0.602659,0.812224,1.059054
4,0.489200,1.052247,0.610999,0.790100,1.025791
5,0.410300,1.151575,0.615798,0.823676,1.073115



FINAL DEV EVALUATION



Dev metrics
Dev Spearman: 0.6158
Dev MAE:      0.8228
Dev RMSE:     1.0728
Dev Acc-SD:   0.7636

FINAL TEST EVALUATION



Test metrics
Test Spearman: 0.5993
Test MAE:      0.8153
Test RMSE:     1.0557
Test Acc-SD:   0.7430
  sample_id  prediction
0      2017    4.758419
1      2018    1.792465
2      2019    4.429935
3      2020    3.489028
4      2021    4.685279

Saved predictions to ./baseline_final/submission.csv

Saved to ./baseline_final/
